# S5 Eval Curves: Offline BC MC vs NAIL-OPD MC

This notebook recreates WandB-style eval plots for three noise regimes:

- low noise
- medium noise
- high noise

Each regime gets two plots:

- `iter` vs `val/clean_full_exact`
- `iter` vs `val/clean_final_exact`

Important workflow note:

- the first few code cells are intentionally lightweight
- W&B syncing happens in one dedicated cell later on
- that sync cell is opt-in: leave `ENABLE_WANDB_SYNC = False` for a fast local run, or flip it to `True` only when you want to refresh cache files


In [ ]:
from pathlib import Path

import json
import re
import sys
import pandas as pd

# Resolve the repo root whether the notebook is opened from the repo root or the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WANDB_ENTITY = "vedsriraman-columbia-university"
WANDB_PROJECT = "small-cot-experiments"
WANDB_CACHE_DIR = ROOT / "analysis" / "cache" / "wandb"
WANDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Keep notebook startup fast by default. Only flip this to True when you explicitly want to refresh W&B cache files.
ENABLE_WANDB_SYNC = False

# Coverage and gap diagnostics are shown for one regime at a time so the notebook stays responsive.
COVERAGE_REGIME = "low"

# Save exported plot images into the repo so they can be embedded in the experiment log markdown.
SAVE_PLOTS = True
PLOT_EXPORT_DIR = ROOT / "analysis" / "figures" / "s5_eval_curves"
PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Offline BC MC runs are sourced from W&B for every eta we want to compare.
OFFLINE_BC_RUNS = {
    0.05: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p05"},
    0.10: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p1"},
    0.20: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p2"},
    0.30: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p3"},
    0.40: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p4"},
    0.50: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p5"},
    0.60: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p6"},
    0.70: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p7"},
    0.80: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p8"},
    0.90: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p9"},
}

# Low-eta NAIL runs need W&B for the early part of training because the local eval file only contains the resumed tail.
# Some of those resumed jobs also have duplicate W&B display names, so we may need to stitch multiple cached W&B histories together.
# For eta >= 0.4 the local eval history on blocklab should already contain the full curve.
NAIL_RUNS = {
    0.05: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p05-distributional_noise-t1p0", "needs_wandb": True},
    0.10: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p1-distributional_noise-t1p0", "needs_wandb": True},
    0.20: {
        "run_name": "s5-opd-forward_kl_simple-n8000000-eta0p2-distributional_noise-t1p0",
        "needs_wandb": True,
        "wandb_run_paths": [
            "vedsriraman-columbia-university/small-cot-experiments/ywtdu7kf",
            "vedsriraman-columbia-university/small-cot-experiments/bk6g7mti",
        ],
    },
    0.30: {
        "run_name": "s5-opd-forward_kl_simple-n8000000-eta0p3-distributional_noise-t1p0",
        "needs_wandb": True,
        "wandb_run_paths": [
            "vedsriraman-columbia-university/small-cot-experiments/7q2vktpm",
            "vedsriraman-columbia-university/small-cot-experiments/lafk67b8",
        ],
    },
    0.40: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p4-distributional_noise-t1p0", "needs_wandb": False},
    0.50: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p5-distributional_noise-t1p0", "needs_wandb": False},
    0.60: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p6-distributional_noise-t1p0", "needs_wandb": False},
    0.70: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p7-distributional_noise-t1p0", "needs_wandb": False},
    0.80: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p8-distributional_noise-t1p0", "needs_wandb": False},
    0.90: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p9-distributional_noise-t1p0", "needs_wandb": False},
}

# Standard OPD-MC corresponds to the native reverse_kl_tm objective.
# The early OPD MC etas also benefit from cached W&B histories because some local eval files are incomplete or absent.
# Later etas are still optional and the per-eta method plots will skip whichever curves do not exist yet.
OPD_RUNS = {
    0.00: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p0-distributional_noise-t1p0", "needs_wandb": True},
    0.05: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p05-distributional_noise-t1p0", "needs_wandb": True},
    0.10: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p1-distributional_noise-t1p0", "needs_wandb": True},
    0.20: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p2-distributional_noise-t1p0", "needs_wandb": True},
    0.30: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p3-distributional_noise-t1p0", "needs_wandb": True},
    0.40: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p4-distributional_noise-t1p0", "needs_wandb": True},
    0.50: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p5-distributional_noise-t1p0", "needs_wandb": True},
    0.60: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p6-distributional_noise-t1p0", "needs_wandb": False},
    0.70: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p7-distributional_noise-t1p0", "needs_wandb": False},
    0.80: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p8-distributional_noise-t1p0", "needs_wandb": False},
    0.90: {"run_name": "s5-opd-reverse_kl_tm-n8000000-eta0p9-distributional_noise-t1p0", "needs_wandb": False},
}

REGIMES = {
    "low": {
        "title_prefix": "low-noise regime",
        "offline_etas": [0.05, 0.10, 0.20, 0.30],
        "nail_etas": [0.05, 0.10, 0.20, 0.30, 0.40],
    },
    "medium": {
        "title_prefix": "medium-noise regime",
        "offline_etas": [0.40, 0.50, 0.60],
        "nail_etas": [0.40, 0.50, 0.60, 0.70],
    },
    "high": {
        "title_prefix": "high-noise regime",
        "offline_etas": [0.70, 0.80, 0.90],
        "nail_etas": [0.70, 0.80, 0.90],
    },
}

ALL_ETAS = sorted(set(OFFLINE_BC_RUNS) | set(NAIL_RUNS) | set(OPD_RUNS))

METRICS = ["val/clean_full_exact", "val/clean_final_exact"]
EVAL_KEYS = ["iter", *METRICS]

ROOT


In [ ]:
# Helper utilities stay local and instant; no network calls happen in this cell.

def to_jsonable(obj):
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        if isinstance(obj, dict):
            return {str(k): to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [to_jsonable(v) for v in obj]
        return str(obj)


def eta_tag(eta: float) -> str:
    return str(eta).replace('.', 'p')


def meta_path_for_run_id(run_id: str) -> Path:
    return WANDB_CACHE_DIR / f"{run_id}.meta.json"


def eval_path_for_run_id(run_id: str) -> Path:
    return WANDB_CACHE_DIR / f"{run_id}.eval.csv"


def build_cached_run_name_lookup() -> dict[str, list[dict]]:
    # Read any already-exported W&B metadata from disk so we can avoid unnecessary API searches.
    # A single display name can map to multiple W&B runs when jobs were resumed or relaunched.
    lookup = {}
    for path in sorted(WANDB_CACHE_DIR.glob("*.meta.json")):
        meta = json.loads(path.read_text(encoding="utf-8"))
        run_name = meta.get("run_name")
        if run_name:
            lookup.setdefault(run_name, []).append(meta)
    for metas in lookup.values():
        metas.sort(
            key=lambda meta: (
                meta.get("history_min_iter", float("inf")),
                meta.get("history_max_iter", -1),
                meta.get("run_id", ""),
            )
        )
    return lookup


def build_cached_run_path_lookup() -> dict[str, dict]:
    lookup = {}
    for path in sorted(WANDB_CACHE_DIR.glob("*.meta.json")):
        meta = json.loads(path.read_text(encoding="utf-8"))
        run_path = meta.get("run_path")
        if run_path:
            lookup[run_path] = meta
    return lookup


def cached_metas_for_info(info: dict, name_lookup: dict[str, list[dict]], path_lookup: dict[str, dict]) -> list[dict]:
    explicit_paths = info.get("wandb_run_paths", [])
    if explicit_paths:
        return [path_lookup[path] for path in explicit_paths if path in path_lookup]
    return name_lookup.get(info["run_name"], [])


def has_required_cached_eval(info: dict, name_lookup: dict[str, list[dict]], path_lookup: dict[str, dict]) -> bool:
    metas = cached_metas_for_info(info, name_lookup, path_lookup)
    explicit_paths = info.get("wandb_run_paths", [])
    if explicit_paths:
        return len(metas) == len(explicit_paths) and all(eval_path_for_run_id(meta["run_id"]).exists() for meta in metas)
    return any(eval_path_for_run_id(meta["run_id"]).exists() for meta in metas)


def describe_missing_cache(info: dict, name_lookup: dict[str, list[dict]], path_lookup: dict[str, dict]) -> str:
    explicit_paths = info.get("wandb_run_paths", [])
    if explicit_paths:
        missing_paths = [
            run_path
            for run_path in explicit_paths
            if run_path not in path_lookup or not eval_path_for_run_id(path_lookup[run_path]["run_id"]).exists()
        ]
        if missing_paths:
            return f"{info['run_name']} missing explicit fragments: {missing_paths}"
    return info["run_name"]


def normalize_eval_df(df: pd.DataFrame) -> pd.DataFrame:
    # Standardize history tables so all downstream code can assume the same schema.
    expected = ["iter", *METRICS]
    missing = [col for col in expected if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    work = df.loc[:, expected].copy()
    work["iter"] = pd.to_numeric(work["iter"], errors="coerce")
    for metric in METRICS:
        work[metric] = pd.to_numeric(work[metric], errors="coerce")
    return work.dropna(subset=["iter"]).reset_index(drop=True)


def dedupe_by_iter(df: pd.DataFrame) -> pd.DataFrame:
    # When we stitch W&B, local logs, and local resumed tails together, repeated iters can appear.
    # Sorting by source_priority means the highest-priority local source wins when multiple sources contain the same iter.
    if df.empty:
        return df.copy()
    work = df.copy().reset_index(drop=True)
    work["_row_order"] = range(len(work))
    work = work.sort_values(["iter", "source_priority", "_row_order"])
    work = work.groupby("iter", as_index=False).last()
    return work.drop(columns=["_row_order"], errors="ignore").sort_values("iter").reset_index(drop=True)


In [ ]:
# Cache status: this is the quick check before any W&B sync work.
cached_lookup = build_cached_run_name_lookup()
cached_path_lookup = build_cached_run_path_lookup()

status_rows = []
for eta, info in OFFLINE_BC_RUNS.items():
    metas = cached_metas_for_info(info, cached_lookup, cached_path_lookup)
    status_rows.append(
        {
            "group": "Offline BC MC",
            "eta": eta,
            "run_name": info["run_name"],
            "n_cached_runs": len(metas),
            "has_cached_meta": bool(metas),
            "has_cached_eval": has_required_cached_eval(info, cached_lookup, cached_path_lookup),
        }
    )

for eta, info in NAIL_RUNS.items():
    metas = cached_metas_for_info(info, cached_lookup, cached_path_lookup)
    status_rows.append(
        {
            "group": "NAIL-OPD MC",
            "eta": eta,
            "run_name": info["run_name"],
            "needs_wandb": info["needs_wandb"],
            "n_cached_runs": len(metas),
            "has_cached_meta": bool(metas),
            "has_cached_eval": has_required_cached_eval(info, cached_lookup, cached_path_lookup),
        }
    )

for eta, info in OPD_RUNS.items():
    metas = cached_metas_for_info(info, cached_lookup, cached_path_lookup)
    status_rows.append(
        {
            "group": "OPD MC",
            "eta": eta,
            "run_name": info["run_name"],
            "needs_wandb": info["needs_wandb"],
            "n_cached_runs": len(metas),
            "has_cached_meta": bool(metas),
            "has_cached_eval": has_required_cached_eval(info, cached_lookup, cached_path_lookup),
        }
    )

pd.DataFrame(status_rows).sort_values(["group", "eta"]).reset_index(drop=True)


In [ ]:
# Optional sync cell: run this once if any required W&B histories are missing from cache.
# This is the only cell that may take a while, because it may search W&B and download history CSVs.

if not ENABLE_WANDB_SYNC:
    print("Skipping W&B sync. Set ENABLE_WANDB_SYNC = True in the config cell when you want to refresh cache files.")
else:
    import wandb

    api = wandb.Api(timeout=120)

    required_infos = list(OFFLINE_BC_RUNS.values()) + [info for info in NAIL_RUNS.values() if info["needs_wandb"]]
    optional_infos = [info for info in OPD_RUNS.values() if info["needs_wandb"]]
    required_run_names = sorted({info["run_name"] for info in required_infos})
    optional_run_names = sorted({info["run_name"] for info in optional_infos})
    explicit_run_paths = sorted(
        {
            run_path
            for info in required_infos + optional_infos
            for run_path in info.get("wandb_run_paths", [])
        }
    )
    names_to_search = set(required_run_names) | set(optional_run_names)
    paths_to_search = set(explicit_run_paths)

    cached_lookup = build_cached_run_name_lookup()
    cached_path_lookup = build_cached_run_path_lookup()

    missing_required = [describe_missing_cache(info, cached_lookup, cached_path_lookup) for info in required_infos if not has_required_cached_eval(info, cached_lookup, cached_path_lookup)]
    missing_optional = [describe_missing_cache(info, cached_lookup, cached_path_lookup) for info in optional_infos if not has_required_cached_eval(info, cached_lookup, cached_path_lookup)]

    print(f"Required W&B runs missing cache: {len(missing_required)}")
    for name in missing_required:
        print("  required:", name)
    print(f"Optional OPD W&B runs missing cache: {len(missing_optional)}")
    for name in missing_optional:
        print("  optional:", name)

    def cache_run(run, force: bool = False) -> None:
        run_path = "/".join(run.path)
        eval_path = eval_path_for_run_id(run.id)
        meta_path = meta_path_for_run_id(run.id)
        if not force and eval_path.exists() and meta_path.exists():
            return

        eval_df = pd.DataFrame(run.scan_history(keys=EVAL_KEYS, page_size=1000))
        eval_df.to_csv(eval_path, index=False)

        iter_values = pd.to_numeric(eval_df.get("iter", pd.Series(dtype=float)), errors="coerce")
        valid_iters = iter_values.dropna()
        meta = {
            "run_id": run.id,
            "run_name": run.name,
            "run_path": run_path,
            "url": run.url,
            "state": run.state,
            "history_rows": int(len(eval_df)),
            "history_min_iter": None if valid_iters.empty else int(valid_iters.min()),
            "history_max_iter": None if valid_iters.empty else int(valid_iters.max()),
            "summary": to_jsonable(dict(run.summary)),
        }
        meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
        print(f"synced {run.name} -> {run.id} ({len(eval_df)} rows)")

    explicit_paths_to_fetch = sorted(
        {
            run_path
            for info in required_infos + optional_infos
            for run_path in info.get("wandb_run_paths", [])
        }
    )
    if explicit_paths_to_fetch:
        print("Refreshing explicit W&B run paths directly...")
        for run_path in explicit_paths_to_fetch:
            print(f"  refreshing {run_path}")
            cache_run(api.run(run_path), force=True)
    else:
        print("All explicit W&B run-path fragments are already cached.")

    cached_lookup = build_cached_run_name_lookup()
    cached_path_lookup = build_cached_run_path_lookup()
    remaining_names_to_search = {
        info["run_name"]
        for info in required_infos + optional_infos
        if not info.get("wandb_run_paths") and not has_required_cached_eval(info, cached_lookup, cached_path_lookup)
    }

    if remaining_names_to_search:
        print("Scanning W&B project for unresolved display names...")
        for run in api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}"):
            if run.name not in remaining_names_to_search:
                continue
            cache_run(run)
    else:
        print("No unresolved display-name lookups remain after explicit fetches.")

    cached_lookup = build_cached_run_name_lookup()
    cached_path_lookup = build_cached_run_path_lookup()
    still_missing_required = [describe_missing_cache(info, cached_lookup, cached_path_lookup) for info in required_infos if not has_required_cached_eval(info, cached_lookup, cached_path_lookup)]
    if still_missing_required:
        raise RuntimeError(f"Could not find these required W&B runs: {sorted(still_missing_required)}")

    print("Cached W&B fragment counts after sync:")
    for info in required_infos + optional_infos:
        metas = cached_metas_for_info(info, cached_lookup, cached_path_lookup)
        expected = len(info.get("wandb_run_paths", []))
        if expected:
            print(f"  {info['run_name']}: {len(metas)}/{expected} cached explicit fragment(s)")
        else:
            print(f"  {info['run_name']}: {len(metas)} cached run(s)")


In [ ]:
# After syncing, rebuild the lookup and define loaders that rely on local cache files.
cached_lookup = build_cached_run_name_lookup()
cached_path_lookup = build_cached_run_path_lookup()
EMPTY_CURVE_COLUMNS = ["iter", *METRICS, "group", "eta", "source", "source_priority", "label"]


def empty_curve_df() -> pd.DataFrame:
    return pd.DataFrame(columns=EMPTY_CURVE_COLUMNS)


def cached_metas_for_spec(info: dict) -> list[dict]:
    return cached_metas_for_info(info, cached_lookup, cached_path_lookup)


LOG_EVAL_RE = re.compile(
    r"(?:eval|final) step (?P<iter>\d+): val loss (?P<val_loss>[-+0-9.eE]+)"
    r"(?:, val cot_exact (?P<cot_exact>[-+0-9.eE]+))?"
    r", val clean_full_exact (?P<clean_full_exact>[-+0-9.eE]+)"
    r", val clean_final_exact (?P<clean_final_exact>[-+0-9.eE]+)"
)


def load_logged_curve(log_path: Path, group: str, eta: float) -> pd.DataFrame:
    if not log_path.exists():
        return empty_curve_df()

    rows = []
    with open(log_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            match = LOG_EVAL_RE.search(line)
            if not match:
                continue
            rows.append(
                {
                    "iter": int(match.group("iter")),
                    "val/clean_full_exact": float(match.group("clean_full_exact")),
                    "val/clean_final_exact": float(match.group("clean_final_exact")),
                }
            )

    if not rows:
        return empty_curve_df()

    df = normalize_eval_df(pd.DataFrame(rows))
    df["group"] = group
    df["eta"] = eta
    df["source"] = str(log_path.relative_to(ROOT))
    df["source_priority"] = 1
    df["label"] = f"{group}, eta={eta:.2f}"
    return dedupe_by_iter(df)


def load_cached_curve_group(info: dict, group: str, eta: float) -> pd.DataFrame:
    frames = []
    for meta in cached_metas_for_spec(info):
        path = eval_path_for_run_id(meta["run_id"])
        if not path.exists():
            continue
        df = normalize_eval_df(pd.read_csv(path))
        df["group"] = group
        df["eta"] = eta
        df["source"] = meta.get("run_path", f"wandb_csv:{meta['run_id']}")
        df["source_priority"] = 0
        df["label"] = f"{group}, eta={eta:.2f}"
        frames.append(df)
    if not frames:
        return empty_curve_df()
    return dedupe_by_iter(pd.concat(frames, ignore_index=True))


def load_offline_bc_curve(eta: float) -> pd.DataFrame:
    # Offline BC MC is sourced entirely from W&B cache.
    if eta not in OFFLINE_BC_RUNS:
        return empty_curve_df()
    return load_cached_curve_group(OFFLINE_BC_RUNS[eta], "Offline BC MC", eta)


def load_local_nail_curve(eta: float) -> pd.DataFrame:
    # Local NAIL eval files are the authoritative source for the resumed tail and for later regimes.
    path = ROOT / f"out-s5-opd-forward_kl_simple-n8000000-eta{eta_tag(eta)}-distributional_noise-t1p0" / "eval_history.jsonl"
    if eta not in NAIL_RUNS:
        return empty_curve_df()
    if not path.exists():
        return empty_curve_df()
    df = normalize_eval_df(pd.read_json(path, lines=True))
    df["group"] = "NAIL-OPD MC"
    df["eta"] = eta
    df["source"] = str(path.relative_to(ROOT))
    df["source_priority"] = 2
    df["label"] = f"NAIL-OPD MC, eta={eta:.2f}"
    return df


def load_logged_nail_curve(eta: float) -> pd.DataFrame:
    if eta not in NAIL_RUNS:
        return empty_curve_df()
    log_path = ROOT / "logs" / "opd" / f"s5_opd_forward_kl_simple_n8000000_eta{eta_tag(eta)}_distributional_noise_t1p0.log"
    return load_logged_curve(log_path, "NAIL-OPD MC", eta)


def load_wandb_nail_curve(eta: float) -> pd.DataFrame:
    # Only the low-eta NAIL runs need W&B history for the early optimizer steps.
    if eta not in NAIL_RUNS or not NAIL_RUNS[eta]["needs_wandb"]:
        return empty_curve_df()
    return load_cached_curve_group(NAIL_RUNS[eta], "NAIL-OPD MC", eta)


def load_nail_curve(eta: float) -> pd.DataFrame:
    # Stitch early W&B history to the local resumed tail, then dedupe repeated iters.
    if eta not in NAIL_RUNS:
        return empty_curve_df()
    combined = pd.concat([load_wandb_nail_curve(eta), load_logged_nail_curve(eta), load_local_nail_curve(eta)], ignore_index=True)
    if combined.empty:
        raise FileNotFoundError(f"No NAIL eval data found for eta={eta}")
    return dedupe_by_iter(combined)


def load_local_opd_curve(eta: float) -> pd.DataFrame:
    # Standard OPD-MC uses the reverse_kl_tm objective and may be missing for some etas while the sweep is still running.
    if eta not in OPD_RUNS:
        return empty_curve_df()
    path = ROOT / f"out-s5-opd-reverse_kl_tm-n8000000-eta{eta_tag(eta)}-distributional_noise-t1p0" / "eval_history.jsonl"
    if not path.exists():
        return empty_curve_df()
    df = normalize_eval_df(pd.read_json(path, lines=True))
    df["group"] = "OPD MC"
    df["eta"] = eta
    df["source"] = str(path.relative_to(ROOT))
    df["source_priority"] = 2
    df["label"] = f"OPD MC, eta={eta:.2f}"
    return df


def load_logged_opd_curve(eta: float) -> pd.DataFrame:
    if eta not in OPD_RUNS:
        return empty_curve_df()
    log_path = ROOT / "logs" / "opd" / f"s5_opd_reverse_kl_tm_n8000000_eta{eta_tag(eta)}_distributional_noise_t1p0.log"
    return load_logged_curve(log_path, "OPD MC", eta)


def load_cached_opd_curve(eta: float) -> pd.DataFrame:
    # If OPD-MC histories have also been exported to W&B cache, use them to fill any missing early steps.
    if eta not in OPD_RUNS:
        return empty_curve_df()
    return load_cached_curve_group(OPD_RUNS[eta], "OPD MC", eta)


def load_opd_curve(eta: float) -> pd.DataFrame:
    combined = pd.concat([load_cached_opd_curve(eta), load_logged_opd_curve(eta), load_local_opd_curve(eta)], ignore_index=True)
    if combined.empty:
        return empty_curve_df()
    return dedupe_by_iter(combined)


def missing_offline_bc_cache_for_regime(regime_key: str) -> list[str]:
    # Offline BC is W&B-cache-only, so missing offline cache should still be a hard error.
    regime = REGIMES[regime_key]
    missing = []
    for eta in regime["offline_etas"]:
        info = OFFLINE_BC_RUNS[eta]
        if not has_required_cached_eval(info, cached_lookup, cached_path_lookup):
            missing.append(describe_missing_cache(info, cached_lookup, cached_path_lookup))
    return missing


def missing_nail_wandb_fragments_for_regime(regime_key: str) -> list[str]:
    # NAIL can fall back to parsed local logs and local eval_history.jsonl, so this is only a warning.
    regime = REGIMES[regime_key]
    missing = []
    for eta in regime["nail_etas"]:
        info = NAIL_RUNS[eta]
        if info["needs_wandb"] and not has_required_cached_eval(info, cached_lookup, cached_path_lookup):
            missing.append(describe_missing_cache(info, cached_lookup, cached_path_lookup))
    return missing


def ensure_regime_cache_ready(regime_key: str) -> None:
    missing = missing_offline_bc_cache_for_regime(regime_key)
    if missing:
        raise RuntimeError(
            f"Missing cached W&B histories for offline BC in regime '{regime_key}': {missing}. "
            "Set ENABLE_WANDB_SYNC = True in the config cell and rerun the sync cell."
        )


def summarize_coverage(curve_df: pd.DataFrame) -> pd.DataFrame:
    return (
        curve_df.groupby(["group", "eta", "label"], as_index=False)
        .agg(
            n_points=("iter", "size"),
            min_iter=("iter", "min"),
            max_iter=("iter", "max"),
            final_clean_full_exact=("val/clean_full_exact", "last"),
            final_clean_final_exact=("val/clean_final_exact", "last"),
        )
        .sort_values(["group", "eta"])
        .reset_index(drop=True)
    )


def summarize_metric_gap(curve_df: pd.DataFrame) -> pd.DataFrame:
    return (
        curve_df.assign(metric_gap=curve_df["val/clean_final_exact"] - curve_df["val/clean_full_exact"])
        .groupby(["group", "eta", "label"], as_index=False)
        .agg(
            max_abs_gap=("metric_gap", lambda s: float(s.abs().max())),
            final_gap=("metric_gap", "last"),
        )
        .sort_values(["group", "eta"])
        .reset_index(drop=True)
    )


curve_df_cache = {}


def load_curve_df_for_regime(regime_key: str) -> pd.DataFrame:
    # Load only the curves needed for one regime so low-noise work does not wait on medium/high cache state.
    if regime_key in curve_df_cache:
        return curve_df_cache[regime_key]

    ensure_regime_cache_ready(regime_key)
    missing_nail_fragments = missing_nail_wandb_fragments_for_regime(regime_key)
    if missing_nail_fragments:
        print(
            f"Warning: proceeding without some cached NAIL W&B fragments for {regime_key} regime: {missing_nail_fragments}. "
            "The notebook will fall back to parsed local logs and local eval_history.jsonl where available."
        )
    regime = REGIMES[regime_key]
    curves = []

    print(f"Loading Offline BC MC curves for {regime_key} regime...")
    for eta in regime["offline_etas"]:
        print(f"  Offline BC MC eta={eta:.2f}")
        curves.append(load_offline_bc_curve(eta))

    print(f"Loading NAIL-OPD MC curves for {regime_key} regime...")
    for eta in regime["nail_etas"]:
        print(f"  NAIL-OPD MC eta={eta:.2f}")
        curves.append(load_nail_curve(eta))

    curve_df = pd.concat(curves, ignore_index=True)
    print(
        f"Loaded {len(curve_df)} eval points across "
        f"{curve_df[['group', 'eta']].drop_duplicates().shape[0]} curves for {regime_key} regime."
    )
    curve_df_cache[regime_key] = curve_df
    return curve_df


In [ ]:
# Coverage summary is shown for one regime at a time.
# That keeps this diagnostic cell fast and avoids failing on medium/high caches when you only care about low noise.
curve_df_for_coverage = load_curve_df_for_regime(COVERAGE_REGIME)
coverage_df = summarize_coverage(curve_df_for_coverage)
coverage_df


In [ ]:
# Diagnostic: quantify how different clean_final_exact is from clean_full_exact.
# In low noise these gaps can be tiny, which makes the two plots look almost identical.
metric_gap_df = summarize_metric_gap(curve_df_for_coverage)
metric_gap_df


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scripts.plot_style import (
    get_method_style,
    polish_axes,
    save_publication_figure,
    set_publication_style,
)

set_publication_style()

# In the regime plots, color encodes eta and line style encodes method.
# That makes matched offline vs online curves at the same eta much easier to eyeball.
ETA_COLORS = {
    0.05: "#F94144",
    0.10: "#F3722C",
    0.20: "#F9C74F",
    0.30: "#90BE6D",
    0.40: "#43AA8B",
    0.50: "#577590",
    0.60: "#277DA1",
    0.70: "#9D4EDD",
    0.80: "#B5179E",
    0.90: "#F72585",
}

# In the per-eta plots, keep one fixed color per method across every figure.
METHOD_STYLE_NAMES = {
    "Offline BC MC": "Offline BC",
    "NAIL-OPD MC": "NAIL-forward",
    "OPD MC": "TM OPD",
}
METHOD_STYLES = {group: get_method_style(method_name) for group, method_name in METHOD_STYLE_NAMES.items()}
METHOD_COLORS = {group: style.color for group, style in METHOD_STYLES.items()}
METHOD_LINESTYLES = {group: style.linestyle for group, style in METHOD_STYLES.items()}
METHOD_MARKERS = {group: style.marker for group, style in METHOD_STYLES.items()}
METHOD_LABELS = {group: style.label for group, style in METHOD_STYLES.items()}


def labels_for_regime(regime_key: str) -> list[str]:
    regime = REGIMES[regime_key]
    return [
        *(f"Offline BC MC, eta={eta:.2f}" for eta in regime["offline_etas"]),
        *(f"NAIL-OPD MC, eta={eta:.2f}" for eta in regime["nail_etas"]),
    ]


def regime_plot_filename(metric: str, regime_key: str) -> str:
    metric_name = metric.split('/')[-1]
    return f"{regime_key}_{metric_name}.png"


def eta_plot_filename(metric: str, eta: float) -> str:
    metric_name = metric.split('/')[-1]
    return f"eta{eta_tag(eta)}_{metric_name}_methods.png"


def plot_regime(metric: str, regime_key: str) -> None:
    # Shared plotting helper so each regime/metric plot cell stays short and readable.
    curve_df = load_curve_df_for_regime(regime_key)
    regime = REGIMES[regime_key]
    labels = labels_for_regime(regime_key)
    metric_name = metric.split('/')[-1]
    pretty_metric = metric_name.replace('_', ' ')
    title = f"{pretty_metric} in {regime['title_prefix']}"

    fig, ax = plt.subplots(figsize=(16, 5.75), constrained_layout=True)
    for label in labels:
        df = curve_df[curve_df["label"] == label].sort_values("iter")
        if df.empty:
            continue
        group = df["group"].iloc[0]
        eta = float(df["eta"].iloc[0])
        ax.plot(
            df["iter"],
            df[metric],
            color=ETA_COLORS[eta],
            linestyle=METHOD_LINESTYLES[group],
            marker=METHOD_MARKERS[group],
            linewidth=METHOD_STYLES[group].linewidth,
            markersize=METHOD_STYLES[group].markersize,
            label=f"{METHOD_LABELS[group]}, eta={eta:.2f}",
        )

    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel(pretty_metric)
    ax.set_ylim(0.0, 1.01)
    ax.xaxis.set_major_locator(MultipleLocator(10000))
    ax.xaxis.set_minor_locator(MultipleLocator(5000))
    polish_axes(ax)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    if SAVE_PLOTS:
        out_path = PLOT_EXPORT_DIR / regime_plot_filename(metric, regime_key)
        save_publication_figure(fig, out_path)
        print(f"Saved plot to {out_path} and {out_path.with_suffix('.pdf')}")

    plt.show()


def plot_eta(metric: str, eta: float) -> None:
    # This alternative view keeps one fixed color per method and gives each eta its own figure.
    metric_name = metric.split('/')[-1]
    pretty_metric = metric_name.replace('_', ' ')
    method_frames = [
        ("Offline BC MC", load_offline_bc_curve(eta)),
        ("NAIL-OPD MC", load_nail_curve(eta)),
        ("OPD MC", load_opd_curve(eta)),
    ]

    fig, ax = plt.subplots(figsize=(14, 5.25), constrained_layout=True)
    missing_methods = []
    for group, df in method_frames:
        if df.empty:
            missing_methods.append(group)
            continue
        ax.plot(
            df["iter"],
            df[metric],
            color=METHOD_COLORS[group],
            linestyle=METHOD_LINESTYLES[group],
            marker=METHOD_MARKERS[group],
            linewidth=METHOD_STYLES[group].linewidth,
            markersize=METHOD_STYLES[group].markersize,
            label=METHOD_LABELS[group],
        )

    ax.set_title(f"{pretty_metric} at eta={eta:.2f}")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(pretty_metric)
    ax.set_ylim(0.0, 1.01)
    ax.xaxis.set_major_locator(MultipleLocator(10000))
    ax.xaxis.set_minor_locator(MultipleLocator(5000))
    polish_axes(ax)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    if SAVE_PLOTS:
        out_path = PLOT_EXPORT_DIR / eta_plot_filename(metric, eta)
        save_publication_figure(fig, out_path)
        print(f"Saved plot to {out_path} and {out_path.with_suffix('.pdf')}")

    if missing_methods:
        print(f"Missing curves for eta={eta:.2f}: {', '.join(missing_methods)}")

    plt.show()


In [ ]:
plot_regime("val/clean_full_exact", "low")


In [ ]:
plot_regime("val/clean_final_exact", "low")


In [ ]:
plot_regime("val/clean_full_exact", "medium")


In [ ]:
plot_regime("val/clean_final_exact", "medium")


In [ ]:
plot_regime("val/clean_full_exact", "high")


In [ ]:
plot_regime("val/clean_final_exact", "high")


In [ ]:
for eta in ALL_ETAS:
    plot_eta("val/clean_full_exact", eta)


In [ ]:
for eta in ALL_ETAS:
    plot_eta("val/clean_final_exact", eta)


In [ ]:
# This view is the quick stitching sanity check for the resumed low-eta NAIL runs.
# It shows the raw W&B fragments, parsed log rows, and the local resumed tail before the final dedupe-by-iter step.
raw_nail_debug_df = pd.concat(
    [
        pd.concat([load_wandb_nail_curve(0.20), load_logged_nail_curve(0.20), load_local_nail_curve(0.20)], ignore_index=True),
        pd.concat([load_wandb_nail_curve(0.30), load_logged_nail_curve(0.30), load_local_nail_curve(0.30)], ignore_index=True),
    ],
    ignore_index=True,
)
raw_nail_debug_df.loc[:, ["eta", "iter", "val/clean_full_exact", "val/clean_final_exact", "source", "source_priority"]].sort_values(["eta", "iter", "source_priority", "source"]).reset_index(drop=True)
